In [ ]:
import torch
import torch.nn as nn
import math


class RNNCell(nn.Module):
    """Mot cell RNN co ban: h_t = tanh(W_ih @ x_t + b_ih + W_hh @ h_{t-1} + b_hh)"""

    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        # Trong so va bias
        self.W_ih = nn.Parameter(torch.empty(hidden_size, input_size))
        self.b_ih = nn.Parameter(torch.zeros(hidden_size))
        self.W_hh = nn.Parameter(torch.empty(hidden_size, hidden_size))
        self.b_hh = nn.Parameter(torch.zeros(hidden_size))

        # Khoi tao trong so theo uniform nhu PyTorch mac dinh
        k = 1.0 / math.sqrt(hidden_size)
        nn.init.uniform_(self.W_ih, -k, k)
        nn.init.uniform_(self.W_hh, -k, k)
        nn.init.uniform_(self.b_ih, -k, k)
        nn.init.uniform_(self.b_hh, -k, k)

    def forward(self, x_t, h_prev):
        # x_t: (batch, input_size), h_prev: (batch, hidden_size)
        return torch.tanh(
            x_t @ self.W_ih.t() + self.b_ih + h_prev @ self.W_hh.t() + self.b_hh
        )


class CustomRNN(nn.Module):
    """RNN nhieu tang (stacked), tuong duong nn.RNN nhung code bang tay."""

    def __init__(self, input_size: int, hidden_size: int, num_layers: int = 1, batch_first: bool = True):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.batch_first = batch_first

        # Tao mot RNNCell cho moi layer
        self.cells = nn.ModuleList()
        for layer in range(num_layers):
            in_sz = input_size if layer == 0 else hidden_size
            self.cells.append(RNNCell(in_sz, hidden_size))

    def forward(self, x, h_0=None):
        # x: (batch, seq_len, input_size) neu batch_first=True
        if self.batch_first:
            x = x.transpose(0, 1)  # -> (seq_len, batch, input_size)

        seq_len, batch_size, _ = x.size()

        if h_0 is None:
            h_0 = torch.zeros(self.num_layers, batch_size, self.hidden_size, device=x.device)

        # h_prev[layer] = hidden state cua layer do tai buoc truoc
        h_prev = [h_0[layer] for layer in range(self.num_layers)]

        outputs = []
        for t in range(seq_len):
            inp = x[t]  # (batch, input_size)
            new_h = []
            for layer, cell in enumerate(self.cells):
                h = cell(inp, h_prev[layer])  # (batch, hidden_size)
                new_h.append(h)
                inp = h  # output cua layer nay la input cua layer ke
            h_prev = new_h
            outputs.append(inp)  # output cua layer cuoi cung

        output = torch.stack(outputs, dim=0)  # (seq_len, batch, hidden_size)
        h_n = torch.stack(h_prev, dim=0)       # (num_layers, batch, hidden_size)

        if self.batch_first:
            output = output.transpose(0, 1)  # -> (batch, seq_len, hidden_size)

        return output, h_n


# --- Kiem thu nhanh ---
rnn = CustomRNN(input_size=10, hidden_size=20, num_layers=2, batch_first=True)
dummy = torch.randn(4, 5, 10)  # batch=4, seq_len=5, input=10
out, h_n = rnn(dummy)
print(f'Output shape : {out.shape}')   # (4, 5, 20)
print(f'Hidden shape : {h_n.shape}')   # (2, 4, 20)
print('Bai 1 OK!')

: 

---
# Bài 2: Phân loại comment Negative / Positive bằng Custom RNN

## 2.1 Import & cấu hình

In [ ]:
import re, csv, random
from pathlib import Path
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from underthesea import word_tokenize

random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2.2 Đọc dữ liệu

In [ ]:
nb_dir  = Path.cwd()
root    = nb_dir.parent
train_dir = root / 'data' / 'data_train' / 'train'
val_dir   = root / 'data' / 'data_train' / 'test'
test_dir  = root / 'data' / 'data_test'  / 'test'

def load_split(split_dir: Path):
    texts, labels = [], []
    label_map = {'neg': 0, 'pos': 1}
    for name in ['neg', 'pos']:
        class_dir = split_dir / name
        if not class_dir.exists():
            continue
        for fp in class_dir.glob('*.txt'):
            texts.append(fp.read_text(encoding='utf-8', errors='ignore'))
            labels.append(label_map[name])
    return texts, labels

train_texts, train_labels = load_split(train_dir)
val_texts,   val_labels   = load_split(val_dir)
test_texts,  test_labels  = load_split(test_dir)

print(f'Train={len(train_texts)} | Val={len(val_texts)} | Test={len(test_texts)}')

## 2.3 Clean, Tokenize bằng underthesea & tạo vocab.csv (tối đa 30 000 từ)

In [ ]:
PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'
MAX_VOCAB = 30000
MAX_LEN   = 180

def clean_tokenize(text: str):
    """Lam sach va tokenize bang underthesea."""
    text = text.strip().lower()
    text = re.sub(r'<[^>]+>', ' ', text)       # loai HTML tag
    text = re.sub(r'http\S+', ' ', text)       # loai URL
    text = re.sub(r'\s+', ' ', text).strip()   # gop khoang trang
    tokens = word_tokenize(text, format='text') # underthesea tra ve chuoi
    return tokens.split()                       # tach thanh list token

# Tokenize toan bo tap train
print('Dang tokenize tap train...')
train_tokenized = [clean_tokenize(t) for t in train_texts]

# Dem tan suat tren tap train
freq = Counter()
for toks in train_tokenized:
    freq.update(toks)

# Lay top MAX_VOCAB tu pho bien nhat
most_common = freq.most_common(MAX_VOCAB)

# Ghi vocab.csv
vocab_path = nb_dir / 'vocab.csv'
with vocab_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['token', 'freq'])
    writer.writerow([PAD_TOKEN, 0])
    writer.writerow([UNK_TOKEN, 0])
    for tok, cnt in most_common:
        writer.writerow([tok, cnt])

# Xay dung word2id tu file vua ghi
vocab_tokens = [PAD_TOKEN, UNK_TOKEN] + [tok for tok, _ in most_common]
word2id = {w: i for i, w in enumerate(vocab_tokens)}
VOCAB_SIZE = len(vocab_tokens)

print(f'Da luu vocab.csv ({VOCAB_SIZE} tokens) tai: {vocab_path}')

## 2.4 Dataset & DataLoader

In [ ]:
def encode(text: str, max_len=MAX_LEN):
    toks = clean_tokenize(text)[:max_len]
    ids = [word2id.get(t, word2id[UNK_TOKEN]) for t in toks]
    length = len(ids)
    ids += [word2id[PAD_TOKEN]] * (max_len - length)
    return ids, max(length, 1)

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.ids, self.lens, self.labels = [], [], labels
        for t in texts:
            i, l = encode(t)
            self.ids.append(i)
            self.lens.append(l)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.ids[idx], dtype=torch.long),
            'length':    torch.tensor(self.lens[idx], dtype=torch.long),
            'label':     torch.tensor(self.labels[idx], dtype=torch.long),
        }

BATCH_SIZE = 64
print('Dang tao dataset...')
train_ds = TextDataset(train_texts, train_labels)
val_ds   = TextDataset(val_texts,   val_labels)
test_ds  = TextDataset(test_texts,  test_labels)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)
print('DataLoader ready!')

## 2.5 Mô hình phân loại sentiment dùng CustomRNN

In [ ]:
EMB_DIM    = 128
HIDDEN_DIM = 128
NUM_LAYERS = 1
NUM_CLASSES = 2

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers, num_classes, pad_idx=0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding  = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.dropout    = nn.Dropout(0.3)
        self.rnn        = CustomRNN(input_size=emb_dim, hidden_size=hidden_dim,
                                    num_layers=num_layers, batch_first=True)
        self.fc         = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids, lengths):
        emb = self.embedding(input_ids)   # (B, T, emb_dim)
        output, h_n = self.rnn(emb)       # output: (B, T, H), h_n: (layers, B, H)
        # Lay hidden state tai vi tri cuoi cung (theo length thuc te)
        # de bo qua cac vi tri padding
        idx = (lengths - 1).long().unsqueeze(1).unsqueeze(2).expand(-1, 1, self.hidden_dim)
        last_hidden = output.gather(1, idx).squeeze(1)   # (B, H)
        return self.fc(self.dropout(last_hidden))

model = SentimentRNN(
    vocab_size=VOCAB_SIZE,
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    pad_idx=word2id[PAD_TOKEN],
).to(device)

print(model)

## 2.6 Huấn luyện & đánh giá

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            lens = batch['length'].to(device)
            labs = batch['label'].to(device)
            logits = model(ids, lens)
            total_loss += criterion(logits, labs).item() * labs.size(0)
            correct += (logits.argmax(1) == labs).sum().item()
            total += labs.size(0)
    return total_loss / total, correct / total

EPOCHS = 20
best_val_acc = 0.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    r_loss, r_correct, r_total = 0.0, 0, 0
    for batch in train_loader:
        ids  = batch['input_ids'].to(device)
        lens = batch['length'].to(device)
        labs = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(ids, lens)
        loss = criterion(logits, labs)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        r_loss += loss.item() * labs.size(0)
        r_correct += (logits.argmax(1) == labs).sum().item()
        r_total += labs.size(0)

    train_loss = r_loss / r_total
    train_acc  = r_correct / r_total
    val_loss, val_acc = evaluate(model, val_loader)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | '
          f'val_loss={val_loss:.4f} | val_acc={val_acc:.4f}')

print(f'\nBest val_acc: {best_val_acc:.4f}')

## 2.7 Đánh giá trên tập Test & lưu model

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

test_loss, test_acc = evaluate(model, test_loader)
print(f'Test loss={test_loss:.4f} | Test acc={test_acc:.4f}')

ckpt_path = nb_dir / 'rnn_custom_sentiment.pt'
torch.save(model.state_dict(), ckpt_path)
print('Saved model:', ckpt_path)